In [192]:

import pyspark
import pandas as pd
import warnings
warnings.filterwarnings(action='ignore')

In [193]:
import findspark
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.feature import StringIndexer, VectorIndexer
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.sql import SparkSession

findspark.init('/Applications/spark-3.3.1-bin-hadoop3')

In [194]:
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName('SparkLab').getOrCreate()

23/01/05 14:19:59 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
23/01/05 14:19:59 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
23/01/05 14:19:59 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
23/01/05 14:19:59 WARN Utils: Service 'SparkUI' could not bind on port 4043. Attempting port 4044.
23/01/05 14:19:59 WARN Utils: Service 'SparkUI' could not bind on port 4044. Attempting port 4045.


In [195]:
data = spark.read.load("./data/Restaurant/df_final_dataset2.csv", 
                       format="csv", sep=",", header="true", inferSchema=True)

In [196]:
data.show()

+-----------+------+----------+-----------+--------+---------------+---------+----+
|customer_id|gender|item_count|grand_total|is_rated|vendor_rating_x|vendor_id|rank|
+-----------+------+----------+-----------+--------+---------------+---------+----+
|          0|  male|         1|        5.2|     Yes|              5|      582|  11|
|          0|  male|         1|        5.2|     Yes|              5|      582|  11|
|          0|  male|         1|        3.5|     Yes|              5|      582|  11|
|          0|  male|         2|        6.3|     Yes|              5|      582|  11|
|          0|  male|         4|       15.0|     Yes|              5|      582|  11|
|          0|  male|         5|       16.0|     Yes|              5|      582|  11|
|          0|  male|         2|        5.7|     Yes|              5|      582|  11|
|          0|  male|         1|        5.2|     Yes|              5|      582|  11|
|          0|  male|         1|        5.2|     Yes|              5|      58

In [183]:
[trainingData, testData] = data.randomSplit([0.7, 0.3])

In [184]:
assembler = VectorAssembler(
    inputCols = ["customer_id", "item_count", "grand_total", "vendor_rating_x", "vendor_id", "rank"],
    outputCol = "features"
)

In [185]:
indexer = StringIndexer(inputCol="vendor_rating_x", outputCol="label")

In [186]:
lr = LogisticRegression(maxIter = 50, regParam = 0.01)

In [187]:
pipeline = Pipeline(stages=[assembler, indexer, lr])

In [188]:
model = pipeline.fit(trainingData)

In [189]:
prediction = model.transform(testData)
prediction.select('vendor_rating_x', 'label', 'probability','prediction').show(truncate=False)

+---------------+-----+-------------------------------------------------------------------------------------------------------+----------+
|vendor_rating_x|label|probability                                                                                            |prediction|
+---------------+-----+-------------------------------------------------------------------------------------------------------+----------+
|5              |0.0  |[0.9502571312565281,0.04234727277123096,0.006472984073354997,7.111990535801902E-5,8.514919935277866E-4]|0.0       |
|5              |0.0  |[0.9502571312565281,0.04234727277123096,0.006472984073354997,7.111990535801902E-5,8.514919935277866E-4]|0.0       |
|5              |0.0  |[0.9502571312565281,0.04234727277123096,0.006472984073354997,7.111990535801902E-5,8.514919935277866E-4]|0.0       |
|5              |0.0  |[0.9502571312565281,0.04234727277123096,0.006472984073354997,7.111990535801902E-5,8.514919935277866E-4]|0.0       |
|5              |0.0  |[0.9

### Evaluating the classifier's accuracy

In [190]:
evaluator = MulticlassClassificationEvaluator(
    labelCol = 'label', predictionCol = 'prediction', metricName='accuracy'
)

accuracy = evaluator.evaluate(prediction)

print("Classification Error = %g" % (1.0 - accuracy))

Classification Error = 0.0785517


In [191]:
spark.stop()